# The Complete Agentic Loop: Movie Agent

In [10]:
import openai
import requests
import json
from dotenv import load_dotenv

load_dotenv()
client = openai.OpenAI()

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

In [11]:
# ── API 호출 함수 ──

# 인기 영호 ㅏ목록
def get_popular_movies():
    res = requests.get(f"{BASE_URL}/movies")
    return res.json()

# ID로 영화 상세 정보 조회
def get_movie_details(id: int):
    res = requests.get(f"{BASE_URL}/movies/{id}")
    return res.json()

# ID로 유사한 영화 조회
def get_similar_movies(id: int):
    res = requests.get(f"{BASE_URL}/movies/{id}/similar")
    return res.json()

# 함수 매핑.
FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [12]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 상세 정보(줄거리, 평점, 개봉일, 장르 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "특정 영화와 비슷한 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer", "description": "영화 ID"}
                },
                "required": ["id"],
            },
        },
    },
]

In [13]:
SYSTEM_PROMPT = """당신은 영화 정보 에이전트입니다.

## 규칙
1. 사용자의 영화 관련 질문에 답하기 위해 제공된 tool을 적극 사용하세요.
2. tool 호출 결과를 바탕으로 자연스러운 한국어로 답변하세요.
3. 필요하면 여러 tool을 순차적으로 호출할 수 있습니다.
4. 영화 ID는 이전 대화에서 나온 값을 활용하세요.
5. 추천할 때는 영화 제목과 함께 ID도 알려주세요.
6. 영화 정보 외의 질문에는 "영화 관련 질문만 도와드릴 수 있습니다"라고 안내하세요.
"""


In [14]:
class MovieAgent:
    def __init__(self):
        self.messages: list[dict] = []

    def _get_api_messages(self) -> list[dict]:
        return [{"role": "system", "content": SYSTEM_PROMPT}] + self.messages

    def _execute_tool_calls(self, tool_calls) -> None:
        """tool_calls 실행 → 결과를 messages에 추가"""
        for tc in tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments) if tc.function.arguments else {}

            print(f"  🔧 {fn_name}({fn_args})")

            fn = FUNCTION_MAP.get(fn_name)
            if fn:
                result = fn(**fn_args)  # 파라미터명 통일 → 언패킹
            else:
                result = {"error": f"Unknown function: {fn_name}"}

            self.messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": fn_name,
                "content": json.dumps(result, ensure_ascii=False),
            })

    def chat(self, user_input: str) -> str:
        """에이전트 루프: tool_calls 있으면 실행 → 재호출 → 최종 응답까지 반복"""
        print(f"User: {user_input}")
        self.messages.append({"role": "user", "content": user_input})

        while True:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                tools=TOOLS,
                messages=self._get_api_messages(),
            )

            choice = response.choices[0]
            message = choice.message

            # assistant 응답 저장
            msg_dict = {"role": "assistant", "content": message.content}
            if message.tool_calls:
                msg_dict["tool_calls"] = [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in message.tool_calls
                ]
            self.messages.append(msg_dict)

            # tool_calls → 실행 후 재호출
            if choice.finish_reason == "tool_calls":
                self._execute_tool_calls(message.tool_calls)
                continue

            # 최종 응답
            answer = message.content or ""
            print(f"Agent: {answer}\n")
            return answer

In [15]:
agent = MovieAgent()

# 1. 인기 영화 조회
agent.chat("지금 인기 있는 영화 알려줘")

User: 지금 인기 있는 영화 알려줘
  🔧 get_popular_movies({})
Agent: 현재 인기 있는 영화 몇 편을 소개합니다.

1. **Your Heart Will Be Broken**
   - **개봉일**: 2026-03-26
   - **줄거리**: 고등학생 폴리나가 새 학교에서 괴롭힘을 피하기 위해 주 bully와 거래를 맺는다. 그 과정에서 진짜 감정이 생기지만, 가족과 친구들이 이들의 사랑을 방해할 이유가 있다.
   - **평점**: 6.99
   - ![포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)

2. **Avatar: Fire and Ash**
   - **개봉일**: 2025-12-17
   - **줄거리**: 제이크 숄리와 네이티리, 그리고 그들의 가족은 새로운 위협에 맞서 싸우게 된다. 그들은 자신의 생존과 판도라의 미래를 위해 싸워야 한다.
   - **평점**: 7.37
   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

3. **Hoppers**
   - **개봉일**: 2026-03-04
   - **줄거리**: 과학자들이 인간의 의식을 로봇 동물에 '홉'하게 함으로써, 동물과 소통할 수 있는 기술을 발견하게 된다.
   - **평점**: 7.58
   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)

4. **The Super Mario Bros. Movie**
   - **개봉일**: 2023-04-05
   - **줄거리**: 브루클린의 배관사 형제인 마리오와 루이지는 신비한 파이프를 통해 마법의 세계로 들어가게 되고, 형제를 찾기 위한 모험을 시작한다.
   - **평점**: 7.59
   - ![포스터](https://image.tmdb.org/t/p/w7

"현재 인기 있는 영화 몇 편을 소개합니다.\n\n1. **Your Heart Will Be Broken**\n   - **개봉일**: 2026-03-26\n   - **줄거리**: 고등학생 폴리나가 새 학교에서 괴롭힘을 피하기 위해 주 bully와 거래를 맺는다. 그 과정에서 진짜 감정이 생기지만, 가족과 친구들이 이들의 사랑을 방해할 이유가 있다.\n   - **평점**: 6.99\n   - ![포스터](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)\n\n2. **Avatar: Fire and Ash**\n   - **개봉일**: 2025-12-17\n   - **줄거리**: 제이크 숄리와 네이티리, 그리고 그들의 가족은 새로운 위협에 맞서 싸우게 된다. 그들은 자신의 생존과 판도라의 미래를 위해 싸워야 한다.\n   - **평점**: 7.37\n   - ![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)\n\n3. **Hoppers**\n   - **개봉일**: 2026-03-04\n   - **줄거리**: 과학자들이 인간의 의식을 로봇 동물에 '홉'하게 함으로써, 동물과 소통할 수 있는 기술을 발견하게 된다.\n   - **평점**: 7.58\n   - ![포스터](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)\n\n4. **The Super Mario Bros. Movie**\n   - **개봉일**: 2023-04-05\n   - **줄거리**: 브루클린의 배관사 형제인 마리오와 루이지는 신비한 파이프를 통해 마법의 세계로 들어가게 되고, 형제를 찾기 위한 모험을 시작한다.\n   - **평점**: 7.59\n   - ![포스터](https://image.tmdb.org/t/p/w780/qNBAXBIQlnOThrVvA6mA2B5ggV6.

In [16]:
# 2. 특정 영화 상세 (이전 대화의 ID 활용)
agent.chat("Avatar: Fire and Ash에 대해 더 알려줘")

User: Avatar: Fire and Ash에 대해 더 알려줘
  🔧 get_movie_details({'id': 83533})
Agent: **Avatar: Fire and Ash**에 대한 자세한 정보는 다음과 같습니다.

- **제목**: Avatar: Fire and Ash
- **개봉일**: 2025-12-17
- **장르**: 과학 소설, 모험, 판타지
- **상영 시간**: 198분
- **예산**: 3억 5천만 달러
- **수익**: 14억 8천만 달러
- **언어**: 영어
- **상태**: 개봉됨
- **평점**: 7.37 (투표 수: 2,514)

**줄거리**: RDA에 대한 파괴적인 전쟁 이후, 제이크 숄리와 네이티리는 새로운 위협인 "Ash People"과 맞서 싸워야 합니다. 이들은 무자비한 바랑(Varang)이 이끄는 폭력적이고 권력을 갈망하는 나비 부족입니다. 제이크의 가족은 생존과 판도라의 미래를 위해 감정적이며 육체적인 한계에 도전하는 전투를 벌여야 합니다.

**태그라인**: "The world of Pandora will change forever."

**제작사**:
- 20th Century Studios
- Lightstorm Entertainment
- TSG Entertainment

**홈페이지**: [Avatar: Fire and Ash 공식 웹사이트](https://www.avatar.com/movies/avatar-fire-and-ash)

![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)

이 영화는 화려한 비주얼과 깊이 있는 스토리로 많은 기대를 모으고 있습니다. 추가적인 정보가 필요하시면 말씀해 주세요!



'**Avatar: Fire and Ash**에 대한 자세한 정보는 다음과 같습니다.\n\n- **제목**: Avatar: Fire and Ash\n- **개봉일**: 2025-12-17\n- **장르**: 과학 소설, 모험, 판타지\n- **상영 시간**: 198분\n- **예산**: 3억 5천만 달러\n- **수익**: 14억 8천만 달러\n- **언어**: 영어\n- **상태**: 개봉됨\n- **평점**: 7.37 (투표 수: 2,514)\n\n**줄거리**: RDA에 대한 파괴적인 전쟁 이후, 제이크 숄리와 네이티리는 새로운 위협인 "Ash People"과 맞서 싸워야 합니다. 이들은 무자비한 바랑(Varang)이 이끄는 폭력적이고 권력을 갈망하는 나비 부족입니다. 제이크의 가족은 생존과 판도라의 미래를 위해 감정적이며 육체적인 한계에 도전하는 전투를 벌여야 합니다.\n\n**태그라인**: "The world of Pandora will change forever."\n\n**제작사**:\n- 20th Century Studios\n- Lightstorm Entertainment\n- TSG Entertainment\n\n**홈페이지**: [Avatar: Fire and Ash 공식 웹사이트](https://www.avatar.com/movies/avatar-fire-and-ash)\n\n![포스터](https://image.tmdb.org/t/p/w780/bRBeSHfGHwkEpImlhxPmOcUsaeg.jpg)\n\n이 영화는 화려한 비주얼과 깊이 있는 스토리로 많은 기대를 모으고 있습니다. 추가적인 정보가 필요하시면 말씀해 주세요!'

In [17]:
# 3. 유사 영화 추천
agent.chat("비슷한 영화 추천해 줄래?")

User: 비슷한 영화 추천해 줄래?
  🔧 get_similar_movies({'id': 83533})
Agent: 다음은 **Avatar: Fire and Ash**와 비슷한 영화 추천 목록입니다.

1. **Puella Magi Madoka Magica the Movie -Walpurgisnacht: Rising-**
   - **개봉일**: 2026-08-28
   - **장르**: 애니메이션, 드라마, 미스터리, 판타지
   - **줄거리**: 'Rebellion' 사건 이후, 평화의 세계가 결코 튼튼하지 않음을 알게 됩니다. 새로운 위협이 등장하며, 주인공 호무라 아케미는 친구를 보호하기 위한 선택의 결과를 감당해야 합니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/g4kCd4rMuN7G3VQQDUT6mLbrBqQ.jpg)

2. **Hocus Pocus 2**
   - **개봉일**: 2022-09-27
   - **장르**: 판타지, 코미디
   - **줄거리**: 29년 만에 되살아난 샌더슨 자매가 새로운 악행을 저지를 준비를 하며, 고등학생들이 이들을 막기 위해 싸웁니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/7ze7YNmUaX81ufctGqt0AgHxRtL.jpg)

3. **Godzilla x Kong: The New Empire**
   - **개봉일**: 2024-03-27
   - **장르**: 액션, 모험, 과학 소설
   - **줄거리**: 고질라와 콩이 새로운 위협에 맞서 힘을 합치는 이야기입니다.
   - ![포스터](https://image.tmdb.org/t/p/w780/z1p34vh7dEOnLDmyCrlUVLuoDzd.jpg)

4. **Green Lantern**
   - **개봉일**: 2011-06-14
   - **장르**: 액션, 모험, 과학 소설
   - **줄거리**: 그린 랜턴 군단의 일원으로서 인간이 우주의 균형을 지키고 새로운

"다음은 **Avatar: Fire and Ash**와 비슷한 영화 추천 목록입니다.\n\n1. **Puella Magi Madoka Magica the Movie -Walpurgisnacht: Rising-**\n   - **개봉일**: 2026-08-28\n   - **장르**: 애니메이션, 드라마, 미스터리, 판타지\n   - **줄거리**: 'Rebellion' 사건 이후, 평화의 세계가 결코 튼튼하지 않음을 알게 됩니다. 새로운 위협이 등장하며, 주인공 호무라 아케미는 친구를 보호하기 위한 선택의 결과를 감당해야 합니다.\n   - ![포스터](https://image.tmdb.org/t/p/w780/g4kCd4rMuN7G3VQQDUT6mLbrBqQ.jpg)\n\n2. **Hocus Pocus 2**\n   - **개봉일**: 2022-09-27\n   - **장르**: 판타지, 코미디\n   - **줄거리**: 29년 만에 되살아난 샌더슨 자매가 새로운 악행을 저지를 준비를 하며, 고등학생들이 이들을 막기 위해 싸웁니다.\n   - ![포스터](https://image.tmdb.org/t/p/w780/7ze7YNmUaX81ufctGqt0AgHxRtL.jpg)\n\n3. **Godzilla x Kong: The New Empire**\n   - **개봉일**: 2024-03-27\n   - **장르**: 액션, 모험, 과학 소설\n   - **줄거리**: 고질라와 콩이 새로운 위협에 맞서 힘을 합치는 이야기입니다.\n   - ![포스터](https://image.tmdb.org/t/p/w780/z1p34vh7dEOnLDmyCrlUVLuoDzd.jpg)\n\n4. **Green Lantern**\n   - **개봉일**: 2011-06-14\n   - **장르**: 액션, 모험, 과학 소설\n   - **줄거리**: 그린 랜턴 군단의 일원으로서 인간이 우주의 균형을 지키고 새로운 적인 파라렉스에 맞서 싸우는 이야기입니다.\n   - ![포스터](http